In [1]:
import os
import gc
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("Libraries imported successfully.")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )



Libraries imported successfully.
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

DATA_DIR = (
    "/kaggle/input/datasets/"
    "taufiqzahantushar/checksent-bn/"
    "CheckSent-BN"
)

TRAIN_PATH = os.path.join(DATA_DIR, "train_data.tsv")
TEST_PATH = os.path.join(DATA_DIR, "test_data.tsv")

MAX_NEW_TOKENS = 10

LABELS = [
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE"
]

print("Model:", MODEL_ID)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)


Model: Qwen/Qwen2.5-3B-Instruct
Train path: /kaggle/input/datasets/taufiqzahantushar/checksent-bn/CheckSent-BN/train_data.tsv
Test path: /kaggle/input/datasets/taufiqzahantushar/checksent-bn/CheckSent-BN/test_data.tsv


In [3]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

print("\nTest sentiment distribution:")
print(test_df["Sentiment"].value_counts())



Train shape: (9254, 6)
Test shape: (2314, 6)

Columns:
['ID', 'Date', 'News Portal', 'Headline', 'Claim', 'Sentiment']

Test sentiment distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64


In [4]:
# ============================================================
# 4. BASIC DATA VALIDATION
# ============================================================

required_columns = [
    "ID",
    "Date",
    "News Portal",
    "Headline",
    "Claim",
    "Sentiment"
]

assert all(
    column in train_df.columns
    for column in required_columns
)

assert all(
    column in test_df.columns
    for column in required_columns
)

assert test_df["Headline"].isna().sum() == 0
assert test_df["Sentiment"].isna().sum() == 0

assert set(test_df["Sentiment"].unique()) == set(LABELS)

print("\nDataset validation passed.")



Dataset validation passed.


In [5]:
# ============================================================
# 5. LOAD HUGGING FACE TOKEN
# ============================================================

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("gptoss")

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [6]:
# ============================================================
# 6. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Tokenizer loaded successfully.")
print(type(tokenizer))


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.
<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [7]:
# ============================================================
# 7. LOAD MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype="auto"
)

model.eval()

print("\nModel loaded successfully.")
print("Model device:", model.device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully.
Model device: cuda:0


In [8]:
# ============================================================
# 8. SENTIMENT PARSER
# ============================================================

def parse_sentiment(output):

    if output is None:
        return None

    output = output.strip().upper()

    # Remove common formatting
    output = output.replace("`", "")
    output = output.replace("*", "")

    # Exact matches first
    if output == "NEGATIVE":
        return "NEGATIVE"

    if output == "NEUTRAL":
        return "NEUTRAL"

    if output == "POSITIVE":
        return "POSITIVE"

    # Search for labels inside generated response
    matches = []

    for label in LABELS:
        if re.search(
            rf"\b{label}\b",
            output
        ):
            matches.append(label)

    # If exactly one label appears
    if len(matches) == 1:
        return matches[0]

    return None

In [9]:
# ============================================================
# 9. ZERO-SHOT PROMPT
# ============================================================

def build_zero_shot_prompt(headline):

    return f"""
You are performing Bengali sentiment classification.

Classify the sentiment of the Bengali news headline into exactly
ONE of the following labels:

NEGATIVE = The headline expresses negative sentiment,
negative events, criticism, conflict, loss, danger, failure,
or other negative emotional polarity.

NEUTRAL = The headline is factual, informational, descriptive,
or does not clearly express positive or negative sentiment.

POSITIVE = The headline expresses positive sentiment,
success, achievement, benefit, celebration, praise,
or other positive emotional polarity.

Important:
- Consider the meaning of the headline.
- Do not classify the Claim field.
- Return ONLY ONE label.
- Do not explain your answer.

Allowed labels:
NEGATIVE
NEUTRAL
POSITIVE

Headline:
{headline}

Label:
""".strip()

In [10]:
# ============================================================
# 10. SINGLE SAMPLE PREDICTION
# ============================================================

def predict_sentiment(headline):

    messages = [
        {
            "role": "user",
            "content": build_zero_shot_prompt(headline)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_tokens = outputs[0, input_length:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(
        generated_text
    )

    return prediction, generated_text

In [11]:
# ============================================================
# 11. TEST A FEW EXAMPLES
# ============================================================

sample_indices = [
    0,
    1,
    2,
    10,
    20
]

print("Sample predictions:\n")

for idx in sample_indices:

    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment(
        headline
    )

    print("=" * 80)
    print("Index:", idx)
    print("Headline:", headline)
    print("Actual:", actual)
    print("Prediction:", prediction)
    print("Raw output:", repr(raw_output))


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sample predictions:

Index: 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Actual: NEUTRAL
Prediction: POSITIVE
Raw output: 'POSITIVE'
Index: 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Actual: POSITIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'


In [12]:
# ============================================================
# 12. RUN ZERO-SHOT ON FULL TEST SET
# ============================================================

zero_shot_predictions = []
zero_shot_raw_outputs = []

print("\nStarting Zero-shot evaluation...\n")

for idx, headline in enumerate(
    tqdm(
        test_df["Headline"],
        total=len(test_df),
        desc="Zero-shot"
    )
):

    try:

        prediction, raw_output = predict_sentiment(
            headline
        )

    except Exception as e:

        print(
            f"\nError at index {idx}: {e}"
        )

        prediction = None
        raw_output = ""

    zero_shot_predictions.append(
        prediction
    )

    zero_shot_raw_outputs.append(
        raw_output
    )

print("\nZero-shot inference completed.")

print(
    "Total predictions:",
    len(zero_shot_predictions)
)

print(
    "Unknown predictions:",
    sum(
        p is None
        for p in zero_shot_predictions
    )
)



Starting Zero-shot evaluation...



Zero-shot:   0%|          | 0/2314 [00:00<?, ?it/s]


Zero-shot inference completed.
Total predictions: 2314
Unknown predictions: 0


In [13]:
# ============================================================
# 13. HANDLE UNKNOWN PREDICTIONS
# ============================================================

y_true = test_df["Sentiment"].tolist()

y_pred = [
    prediction if prediction in LABELS
    else "UNKNOWN"
    for prediction in zero_shot_predictions
]

print("\nPrediction distribution:")
print(
    pd.Series(y_pred).value_counts()
)



Prediction distribution:
NEGATIVE    1759
POSITIVE     517
NEUTRAL       38
Name: count, dtype: int64


In [14]:
# ============================================================
# 14. CALCULATE MAIN METRICS
# ============================================================

# Only calculate metrics on valid predictions.
# Unknown outputs are counted separately.

valid_indices = [
    i
    for i, prediction in enumerate(y_pred)
    if prediction in LABELS
]

valid_y_true = [
    y_true[i]
    for i in valid_indices
]

valid_y_pred = [
    y_pred[i]
    for i in valid_indices
]

accuracy = accuracy_score(
    valid_y_true,
    valid_y_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        valid_y_true,
        valid_y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
)

In [15]:
# ============================================================
# 15. DISPLAY MAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("QWEN2.5-3B-INSTRUCT — ZERO-SHOT RESULTS")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision: {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall   : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"Macro F1 : {macro_f1:.4f} "
    f"({macro_f1 * 100:.2f}%)"
)

print(
    f"Unknown  : {len(y_true) - len(valid_y_true)}"
)


QWEN2.5-3B-INSTRUCT — ZERO-SHOT RESULTS
Accuracy : 0.5834 (58.34%)
Precision: 0.6790 (67.90%)
Recall   : 0.5471 (54.71%)
Macro F1 : 0.4538 (45.38%)
Unknown  : 0


In [16]:
# ============================================================
# 16. CLASS-WISE RESULTS
# ============================================================

print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        valid_y_true,
        valid_y_pred,
        labels=LABELS,
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)




CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.6146    0.9626    0.7502      1123
     NEUTRAL     0.9737    0.0448    0.0857       825
    POSITIVE     0.4487    0.6339    0.5255       366

    accuracy                         0.5834      2314
   macro avg     0.6790    0.5471    0.4538      2314
weighted avg     0.7164    0.5834    0.4777      2314



In [17]:
# ============================================================
# 17. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    valid_y_true,
    valid_y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print("\nConfusion Matrix:")
display(cm_df)



Confusion Matrix:


,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,1081,1,41
NEUTRAL,544,37,244
POSITIVE,134,0,232


In [18]:
# ============================================================
# 18. FINAL RESULT TABLE
# ============================================================

zero_shot_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        macro_f1
    ]
})

print("\nFinal Zero-shot Results:")
display(zero_shot_results)


Final Zero-shot Results:


,Metric,Score
0,Accuracy,0.583405
1,Precision,0.678994
2,Recall,0.547109
3,Macro F1,0.453801
